# 02 - Features intra-anuales históricas y futuras

Este notebook genera las variables intra-anuales usadas por el modelo de seguro.

**Entrada principal**

- `data/processed/clima_mes_2005-2027.csv`

**Entradas auxiliares**

- `data/processed/produccion_caldas_municipal_2007-2024.csv`
- `data/processed/municipios.csv`
- `data/model/clusters_municipales.csv` *(si existe)*

**Salidas**

- `data/processed/features_intra_anuales_2007-2024.csv`
- `data/model/features_intra_anuales_2025-2027.csv`

La misma función genera features históricas y futuras, garantizando consistencia de columnas.


## 1. Librerías

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


## 2. Rutas

In [ ]:
# ============================================================
# RUTAS ROBUSTAS DEL PROYECTO
# ============================================================

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

PATH_PROCESSED = PROJECT_ROOT / "data" / "processed"
PATH_MODEL = PROJECT_ROOT / "data" / "model"

PATH_MODEL.mkdir(parents=True, exist_ok=True)

# Entradas
FILE_CLIMA_MES = PATH_PROCESSED / "clima_mes_2005-2027.csv"
FILE_PRODUCCION = PATH_PROCESSED / "produccion_caldas_municipal_2007-2024.csv"
FILE_MUNICIPIOS = PATH_PROCESSED / "municipios.csv"
FILE_CLUSTERS = PATH_MODEL / "clusters_municipales.csv"

# Salidas
OUTPUT_FEATURES_HIST = PATH_PROCESSED / "features_intra_anuales_2007-2024.csv"
OUTPUT_FEATURES_FUT = PATH_MODEL / "features_intra_anuales_2025-2027.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FILE_CLIMA_MES:", FILE_CLIMA_MES)
print("OUTPUT_FEATURES_HIST:", OUTPUT_FEATURES_HIST)
print("OUTPUT_FEATURES_FUT:", OUTPUT_FEATURES_FUT)


## 3. Carga de datos

In [ ]:
clima_mes = pd.read_csv(FILE_CLIMA_MES)

clima_mes["date"] = pd.to_datetime(clima_mes["date"], errors="coerce")
clima_mes["anio"] = clima_mes["anio"].astype(int)
clima_mes["mes"] = clima_mes["mes"].astype(int)

clima_mes = clima_mes.sort_values(["municipio", "date"]).reset_index(drop=True)

print("Clima mensual:", clima_mes.shape)
print("Rango:", clima_mes["anio"].min(), "-", clima_mes["anio"].max())
print("Municipios:", clima_mes["municipio"].nunique())

clima_mes.head()


In [ ]:
# Producción histórica
produccion = pd.read_csv(FILE_PRODUCCION)

produccion = produccion.rename(columns={
    "Municipio": "MP_NOMBRE",
    "MpNombre": "municipio",
    "Año": "anio",
    "Producción (t)": "produccion_t"
})

if "date" in produccion.columns:
    produccion["date"] = pd.to_datetime(produccion["date"], errors="coerce")

prod_cols = [
    "municipio", "date", "anio", "Área sembrada (ha)",
    "Rendimiento (t/ha)", "produccion_t"
]

prod = produccion[[c for c in prod_cols if c in produccion.columns]].copy()
prod["anio"] = prod["anio"].astype(int)

print("Producción:", prod.shape)
prod.head()


In [ ]:
# Altitud municipal, si no está ya en clima_mes
if "MpAltitud" not in clima_mes.columns:
    municipios = pd.read_csv(FILE_MUNICIPIOS)
    municipios = municipios.rename(columns={"MpNombre": "municipio"})
    mun = municipios[["municipio", "MpAltitud"]].copy()

    clima_mes = clima_mes.merge(mun, on="municipio", how="left")

if clima_mes["MpAltitud"].isna().sum() > 0:
    print("Advertencia: hay registros sin MpAltitud")

print("MpAltitud disponible:", "MpAltitud" in clima_mes.columns)


## 4. Variables base esperadas

In [ ]:
vars_base_obligatorias = [
    "precip_mm",
    "precip_mean_mm",
    "precip_max_mm",
    "temp_mean",
    "temp_min",
    "temp_max",
    "et_real_mm",
    "et_potencial_mm",
    "ndvi_mean",
    "ndvi_min",
    "ndvi_max"
]

faltantes = [v for v in vars_base_obligatorias if v not in clima_mes.columns]

if faltantes:
    raise ValueError(f"Faltan variables climáticas base en clima_mes: {faltantes}")

for v in vars_base_obligatorias:
    clima_mes[v] = pd.to_numeric(clima_mes[v], errors="coerce")

print("Variables base disponibles")


## 5. Funciones de ingeniería de variables

La función `construir_features_intra_anuales` replica la lógica histórica y se aplica sobre todo el periodo 2005–2027.

La climatología para extremos y z-scores se calcula usando solo años históricos hasta 2024.


In [ ]:
def max_streak(values):
    values = pd.Series(values).fillna(0).astype(int).to_numpy()
    max_run = 0
    current = 0

    for value in values:
        if value == 1:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0

    return max_run


def flatten_columns(df):
    df.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple)
        else str(col)
        for col in df.columns
    ]
    return df


In [ ]:
def construir_features_intra_anuales(clima_mes, anio_climatologia_max=2024):

    clima_feat_mes = clima_mes.copy()

    # --------------------------------------------------------
    # Variables derivadas mensuales base
    # --------------------------------------------------------
    clima_feat_mes["balance_hidrico_mm"] = (
        clima_feat_mes["precip_mm"] - clima_feat_mes["et_real_mm"]
    )

    clima_feat_mes["deficit_hidrico_mm"] = (
        clima_feat_mes["et_potencial_mm"] - clima_feat_mes["precip_mm"]
    ).clip(lower=0)

    clima_feat_mes["exceso_hidrico_mm"] = (
        clima_feat_mes["precip_mm"] - clima_feat_mes["et_potencial_mm"]
    ).clip(lower=0)

    clima_feat_mes["ratio_etp_precip"] = (
        clima_feat_mes["et_potencial_mm"] /
        clima_feat_mes["precip_mm"].replace(0, np.nan)
    )

    clima_feat_mes["ratio_etr_etp"] = (
        clima_feat_mes["et_real_mm"] /
        clima_feat_mes["et_potencial_mm"].replace(0, np.nan)
    )

    # --------------------------------------------------------
    # Climatología histórica municipio-mes
    # --------------------------------------------------------
    vars_extremos = [
        "precip_mm", "precip_mean_mm", "precip_max_mm",
        "temp_mean", "temp_min", "temp_max",
        "et_real_mm", "et_potencial_mm",
        "ndvi_mean", "ndvi_min", "ndvi_max",
        "balance_hidrico_mm", "deficit_hidrico_mm", "exceso_hidrico_mm"
    ]

    clima_hist = clima_feat_mes[clima_feat_mes["anio"] <= anio_climatologia_max].copy()

    climatologia = (
        clima_hist
        .groupby(["municipio", "mes"])[vars_extremos]
        .agg(["mean", "std", lambda x: x.quantile(0.10), lambda x: x.quantile(0.90)])
    )

    climatologia.columns = [
        f"{var}_{stat}".replace("<lambda_0>", "p10").replace("<lambda_1>", "p90")
        for var, stat in climatologia.columns
    ]

    climatologia = climatologia.reset_index()

    clima_feat_mes = clima_feat_mes.merge(
        climatologia,
        on=["municipio", "mes"],
        how="left",
        suffixes=("", "_clim")
    )

    # --------------------------------------------------------
    # Z-scores respecto a climatología histórica
    # --------------------------------------------------------
    for v in vars_extremos:
        mean_col = f"{v}_mean"
        std_col = f"{v}_std"

        clima_feat_mes[f"{v}_z"] = (
            (clima_feat_mes[v] - clima_feat_mes[mean_col]) /
            clima_feat_mes[std_col]
        )

        clima_feat_mes[f"{v}_z"] = (
            clima_feat_mes[f"{v}_z"]
            .replace([np.inf, -np.inf], np.nan)
        )

    # --------------------------------------------------------
    # Flags de extremos
    # --------------------------------------------------------
    flags_extremos = [
        "flag_precip_baja_p10",
        "flag_precip_alta_p90",
        "flag_deficit_alto_p90",
        "flag_exceso_alto_p90",
        "flag_temp_media_alta_p90",
        "flag_temp_media_baja_p10",
        "flag_temp_max_alta_p90",
        "flag_temp_min_baja_p10",
        "flag_ndvi_bajo_p10",
        "flag_ndvi_alto_p90"
    ]

    clima_feat_mes["flag_precip_baja_p10"] = (
        clima_feat_mes["precip_mm"] <= clima_feat_mes["precip_mm_p10"]
    ).astype(int)

    clima_feat_mes["flag_precip_alta_p90"] = (
        clima_feat_mes["precip_mm"] >= clima_feat_mes["precip_mm_p90"]
    ).astype(int)

    clima_feat_mes["flag_deficit_alto_p90"] = (
        clima_feat_mes["deficit_hidrico_mm"] >= clima_feat_mes["deficit_hidrico_mm_p90"]
    ).astype(int)

    clima_feat_mes["flag_exceso_alto_p90"] = (
        clima_feat_mes["exceso_hidrico_mm"] >= clima_feat_mes["exceso_hidrico_mm_p90"]
    ).astype(int)

    clima_feat_mes["flag_temp_media_alta_p90"] = (
        clima_feat_mes["temp_mean"] >= clima_feat_mes["temp_mean_p90"]
    ).astype(int)

    clima_feat_mes["flag_temp_media_baja_p10"] = (
        clima_feat_mes["temp_mean"] <= clima_feat_mes["temp_mean_p10"]
    ).astype(int)

    clima_feat_mes["flag_temp_max_alta_p90"] = (
        clima_feat_mes["temp_max"] >= clima_feat_mes["temp_max_p90"]
    ).astype(int)

    clima_feat_mes["flag_temp_min_baja_p10"] = (
        clima_feat_mes["temp_min"] <= clima_feat_mes["temp_min_p10"]
    ).astype(int)

    clima_feat_mes["flag_ndvi_bajo_p10"] = (
        clima_feat_mes["ndvi_mean"] <= clima_feat_mes["ndvi_mean_p10"]
    ).astype(int)

    clima_feat_mes["flag_ndvi_alto_p90"] = (
        clima_feat_mes["ndvi_mean"] >= clima_feat_mes["ndvi_mean_p90"]
    ).astype(int)

    # --------------------------------------------------------
    # Agregados anuales
    # --------------------------------------------------------
    agg_anual = (
        clima_feat_mes
        .groupby(["municipio", "anio"])
        .agg({
            "precip_mm": ["sum", "mean", "std", "min", "max"],
            "precip_mean_mm": ["mean", "std", "min", "max"],
            "precip_max_mm": ["max", "mean"],

            "temp_mean": ["mean", "std", "min", "max"],
            "temp_min": ["min", "mean"],
            "temp_max": ["max", "mean"],

            "et_real_mm": ["sum", "mean", "std", "min", "max"],
            "et_potencial_mm": ["sum", "mean", "std", "min", "max"],

            "balance_hidrico_mm": ["sum", "mean", "min", "max"],
            "deficit_hidrico_mm": ["sum", "mean", "max"],
            "exceso_hidrico_mm": ["sum", "mean", "max"],
            "ratio_etp_precip": ["mean", "max"],
            "ratio_etr_etp": ["mean", "min"],

            "ndvi_mean": ["mean", "std", "min", "max"],
            "ndvi_min": ["min", "mean"],
            "ndvi_max": ["max", "mean"],

            "MpAltitud": "first"
        })
    )

    agg_anual = flatten_columns(agg_anual).reset_index()
    agg_anual = agg_anual.rename(columns={"MpAltitud_first": "MpAltitud"})

    # --------------------------------------------------------
    # Conteo anual de extremos y rachas máximas
    # --------------------------------------------------------
    extremos_anual = (
        clima_feat_mes
        .groupby(["municipio", "anio"])[flags_extremos]
        .sum()
        .reset_index()
    )

    rachas_anual = (
        clima_feat_mes
        .sort_values(["municipio", "anio", "mes"])
        .groupby(["municipio", "anio"])[flags_extremos]
        .agg(max_streak)
        .reset_index()
    )

    rachas_anual = rachas_anual.rename(
        columns={c: f"racha_max_{c}" for c in flags_extremos}
    )

    # --------------------------------------------------------
    # Ventanas intra-anuales
    # --------------------------------------------------------
    ventanas = {
        "m01_03": [1, 2, 3],
        "m04_06": [4, 5, 6],
        "m07_09": [7, 8, 9],
        "m10_12": [10, 11, 12],
        "m01_06": [1, 2, 3, 4, 5, 6],
        "m07_12": [7, 8, 9, 10, 11, 12],
    }

    vars_ventanas = [
        "precip_mm", "temp_mean", "temp_max",
        "et_real_mm", "et_potencial_mm",
        "balance_hidrico_mm", "deficit_hidrico_mm",
        "exceso_hidrico_mm", "ndvi_mean"
    ]

    bloques_ventanas = []

    for nombre_ventana, meses in ventanas.items():
        tmp = clima_feat_mes[clima_feat_mes["mes"].isin(meses)].copy()

        agg_tmp = (
            tmp
            .groupby(["municipio", "anio"])[vars_ventanas]
            .agg(["sum", "mean", "min", "max"])
        )

        agg_tmp.columns = [
            f"{var}_{stat}_{nombre_ventana}"
            for var, stat in agg_tmp.columns
        ]

        bloques_ventanas.append(agg_tmp.reset_index())

    ventanas_anual = bloques_ventanas[0]

    for bloque in bloques_ventanas[1:]:
        ventanas_anual = ventanas_anual.merge(
            bloque,
            on=["municipio", "anio"],
            how="outer"
        )

    # --------------------------------------------------------
    # Variables mensuales explícitas var_m01 ... var_m12
    # --------------------------------------------------------
    vars_pivot = [
        "precip_mm", "precip_mean_mm", "precip_max_mm",
        "temp_mean", "temp_min", "temp_max",
        "et_real_mm", "et_potencial_mm",
        "balance_hidrico_mm", "deficit_hidrico_mm", "exceso_hidrico_mm",
        "ndvi_mean", "ndvi_min", "ndvi_max"
    ]

    mensual_wide = clima_feat_mes.pivot_table(
        index=["municipio", "anio"],
        columns="mes",
        values=vars_pivot,
        aggfunc="mean"
    )

    mensual_wide.columns = [
        f"{var}_m{int(mes):02d}"
        for var, mes in mensual_wide.columns
    ]

    mensual_wide = mensual_wide.reset_index()

    # --------------------------------------------------------
    # Anomalías anuales
    # --------------------------------------------------------
    vars_z = [c for c in clima_feat_mes.columns if c.endswith("_z")]

    anomalias_anual = (
        clima_feat_mes
        .groupby(["municipio", "anio"])[vars_z]
        .agg(["mean", "min", "max"])
    )

    anomalias_anual.columns = [
        f"{var}_{stat}_anual"
        for var, stat in anomalias_anual.columns
    ]

    anomalias_anual = anomalias_anual.reset_index()

    # --------------------------------------------------------
    # Unión final
    # --------------------------------------------------------
    features = agg_anual.copy()

    for bloque in [extremos_anual, rachas_anual, ventanas_anual, mensual_wide, anomalias_anual]:
        features = features.merge(
            bloque,
            on=["municipio", "anio"],
            how="left"
        )

    return features, clima_feat_mes


## 6. Generar features intra-anuales 2005–2027

In [ ]:
features_all, clima_feat_mes = construir_features_intra_anuales(
    clima_mes,
    anio_climatologia_max=2024
)

print("Features generadas:", features_all.shape)
print("Años:", features_all["anio"].min(), "-", features_all["anio"].max())
features_all.head()


## 7. Separar histórico y futuro

In [ ]:
# Histórico para selección/modelado: se une con producción observada 2007-2024
features_hist = features_all[
    (features_all["anio"] >= 2007) &
    (features_all["anio"] <= 2024)
].copy()

features_hist = features_hist.merge(
    prod,
    on=["municipio", "anio"],
    how="left"
)

# Reordenar columnas base al inicio
cols_inicio_hist = [
    "municipio", "date", "anio", "Área sembrada (ha)",
    "Rendimiento (t/ha)", "produccion_t"
]

cols_inicio_hist = [c for c in cols_inicio_hist if c in features_hist.columns]
cols_resto_hist = [c for c in features_hist.columns if c not in cols_inicio_hist]

features_hist = features_hist[cols_inicio_hist + cols_resto_hist]

# Futuro para modelo operativo
features_fut = features_all[
    (features_all["anio"] >= 2025) &
    (features_all["anio"] <= 2027)
].copy()

print("Histórico:", features_hist.shape)
print("Futuro:", features_fut.shape)


## 8. Agregar clúster a features futuras

In [ ]:
if FILE_CLUSTERS.exists():
    clusters_mun = pd.read_csv(FILE_CLUSTERS)

    if "cluster" in features_fut.columns:
        features_fut = features_fut.drop(columns=["cluster"])

    features_fut = features_fut.merge(
        clusters_mun[["municipio", "cluster"]],
        on="municipio",
        how="left"
    )

    if features_fut["cluster"].isna().sum() > 0:
        raise ValueError("Hay features futuras sin clúster asignado.")

    features_fut["cluster"] = features_fut["cluster"].astype(int)

    print("Cluster agregado a features futuras")
else:
    print("Advertencia: no se encontró clusters_municipales.csv")


## 9. Validación de compatibilidad con variables del índice

In [ ]:
FILE_VARIABLES_INDICE = PATH_MODEL / "variables_indice_final.csv"

if FILE_VARIABLES_INDICE.exists():
    variables_indice = pd.read_csv(FILE_VARIABLES_INDICE)["variable"].dropna().tolist()

    faltantes_fut = sorted(set(variables_indice) - set(features_fut.columns))

    if faltantes_fut:
        raise ValueError(f"Variables del índice faltantes en features futuras: {faltantes_fut}")

    print("Validación OK: features futuras contienen todas las variables del índice.")
else:
    print("Advertencia: aún no existe variables_indice_final.csv; se omite validación.")


## 10. Guardar salidas

In [ ]:
features_hist.to_csv(OUTPUT_FEATURES_HIST, index=False)
features_fut.to_csv(OUTPUT_FEATURES_FUT, index=False)

print("Archivos guardados:")
print(OUTPUT_FEATURES_HIST)
print(OUTPUT_FEATURES_FUT)


## 11. Siguiente paso

Después de ejecutar este notebook:

1. Si se actualizó `features_intra_anuales_2007-2024.csv`, debe re-ejecutarse:
   - `03_clusters_municipales.ipynb`
   - `04_seleccion_features.ipynb`
   - `05_indice_riesgo.ipynb`

2. Si solo se actualizó `features_intra_anuales_2025-2027.csv`, se puede continuar directamente con:
   - `07_modelo_seguro.ipynb`
